In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (classification_report, roc_curve, auc,
                              mean_squared_error, r2_score)
from sklearn.model_selection import cross_val_score

(X_train, X_test, X_train_sc, X_test_sc,
 y_train_r, y_test_r, y_train_c, y_test_c,
 feature_names) = joblib.load("../data/processed/splits.pkl")

In [2]:
import glob
reg_path = glob.glob("../models/best_regressor_*.pkl")[0]
clf_path = glob.glob("../models/best_classifier_*.pkl")[0]
best_reg = joblib.load(reg_path)
best_clf = joblib.load(clf_path)
print(f"Regresión: {reg_path}")
print(f"Clasificación: {clf_path}")

IndexError: list index out of range

In [ ]:
import glob
reg_path = glob.glob("../models/best_regressor_*.pkl")[0]
clf_path = glob.glob("../models/best_classifier_*.pkl")[0]
best_reg = joblib.load(reg_path)
best_clf = joblib.load(clf_path)
print(f"Regresión: {reg_path}")
print(f"Clasificación: {clf_path}")

In [ ]:

cv_r2 = cross_val_score(best_reg, X_train, y_train_r, cv=5, scoring="r2")
print(f"\nCV Regresión R²: {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")

In [ ]:

cv_f1 = cross_val_score(best_clf, X_train, y_train_c, cv=5, scoring="f1")
print(f"CV Clasificación F1: {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")

In [ ]:
y_pred_c = best_clf.predict(X_test)
print("\n=== Reporte de Clasificación (Test) ===")
print(classification_report(y_test_c, y_pred_c, target_names=["No Viral","Viral"]))

In [ ]:
y_proba = best_clf.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test_c, y_proba)
roc_auc = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="#1E88E5", lw=2, label=f"ROC AUC = {roc_auc:.3f}")
ax.plot([0,1],[0,1], "k--")
ax.set_xlabel("Tasa de Falsos Positivos")
ax.set_ylabel("Tasa de Verdaderos Positivos")
ax.set_title("Curva ROC — Modelo de Clasificación")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("../reports/figures/roc_curve.png", dpi=150)
plt.show()

In [ ]:
y_pred_r = best_reg.predict(X_test)
residuals = y_test_r.values - y_pred_r
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_pred_r, residuals, alpha=0.3, s=10, color="#EF5350")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Residuales vs Predichos")
axes[0].set_xlabel("Predicho"); axes[0].set_ylabel("Residual")
axes[1].hist(residuals, bins=40, color="#42A5F5", edgecolor="white")
axes[1].set_title("Distribución de Residuales")
plt.tight_layout()
plt.savefig("../reports/figures/residuals.png", dpi=150)
plt.show()

In [ ]:
print(f"║ Regresión  R²   (test):  {r2_score(y_test_r, y_pred_r):.4f}       ║")
print(f"║ Regresión  RMSE (test):  {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.4f}       ║")
print(f"║ CV R² (5-fold):          {cv_r2.mean():.4f} ± {cv_r2.std():.3f}  ║")
print(f"║ Clasificación F1 (test): {cv_f1.mean():.4f}       ║")
print(f"║ ROC AUC (test):          {roc_auc:.4f}       ║")